<table align="center">
  <td align="center"><a target="_blank" href="https://github.com/Armstrong66/computer-vision-tutorials">
        <img src="https://img.icons8.com/color/96/000000/artificial-intelligence.png" style="padding-bottom:5px; height:50px;" />
      <br/><b>Computer Vision Tutorials</b></a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/Armstrong66/computer-vision-tutorials/blob/main/tutorials/01_how_neural_networks_see/01_how_neural_networks_see_images.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" /><br/><b>Run in Google Colab</b></a></td>
  <td align="center"><a target="_blank" href="https://github.com/Armstrong66/computer-vision-tutorials">
        <img src="https://i.ibb.co/xfJbPmL/github.png" height="50px" style="padding-bottom:5px;" /><br/><b>View Source on GitHub</b></a></td>
</table>

# Copyright Information


In [ ]:
# Copyright 2026 Armstrong66. Computer Vision Tutorials. All Rights Reserved.
#
# Licensed under the MIT License.
# Repository: https://github.com/Armstrong66/computer-vision-tutorials
#


# Tutorial 01: Computer Vision & Visual Representation
# How Neural Networks "See" Images: Convolutions, Channels, and Feature Hierarchies

Welcome to this interactive tutorial on Computer Vision! Modern deep learning frameworks make it trivially easy to write `tf.keras.layers.Conv2D(32, (3, 3))`, but to build real intuition we need to look beneath the surface.

### Laboratory Roadmap:
1. **The Anatomy of Images & Channels**: Tensors, 1/2/3/N-channel representations, pairwise channel comparisons, and preprocessing.
2. **The Mechanics of Convolutions**: Classical filters, pure NumPy sliding windows, numerical equivalence with TensorFlow.
3. **Spatial Transformations**: Patch formation, padding mechanics, and pooling strategies.
4. **Hierarchies of Vision**: Low-level vs. high-level features, non-linearity (ReLU), and intermediate activation maps.
5. **Interactive Exercises (TODOs)**: Hands-on coding challenges and sandbox exploration.


## 0.0 Setup and Dependencies


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os
from PIL import Image

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['image.interpolation'] = 'nearest'

print(f"TensorFlow Version: {tf.__version__}")
gpu_devices = tf.config.list_physical_devices('GPU')
print(f"GPU Available: {len(gpu_devices) > 0} ({gpu_devices if gpu_devices else 'Running on CPU'})")


---
# Part 1: How Computers "See" Images — Tensors and Channels

To a computer and a neural network, an image is a **multidimensional numerical tensor**:
$$\mathbf{X} \in \mathbb{R}^{H \times W \times C}$$

### What do different channel counts ($C$) represent?
| $C$ | Representation | Examples |
|:---:|:---|:---|
| 1 | Grayscale scalar intensity | MNIST digits, X-rays |
| 2 | 2D vector field | Optical flow, image gradients $(\partial I/\partial x,\; \partial I/\partial y)$ |
| 3 | RGB additive color | Standard digital photographs |
| $\ge 4$ | Multi-spectral / latent features | Satellite imagery, CNN intermediate feature maps |


In [ ]:
def load_sample_image():
    """Load a sample image with local, remote, and synthetic fallbacks."""
    # 1. Local paths
    for path in ['data/sample.jpg', 'sample_image.jpg']:
        if os.path.exists(path):
            return np.array(Image.open(path).convert('RGB'))

    # 2. Remote fallback — download from this repository
    url = "https://raw.githubusercontent.com/Armstrong66/computer-vision-tutorials/main/data/sample.jpg"
    try:
        urllib.request.urlretrieve(url, "sample_image.jpg")
        return np.array(Image.open("sample_image.jpg").convert('RGB'))
    except Exception:
        pass

    # 3. Synthetic fallback — vibrant color regions for clear demos
    print("Creating synthetic demo image (no sample found)...")
    img = np.full((256, 256, 3), 180, dtype=np.uint8)  # light-gray base
    # Red block (top-left)
    img[20:110, 20:110, :] = [220, 40, 40]
    # Green block (top-right)
    img[20:110, 146:236, :] = [40, 200, 40]
    # Blue block (bottom-left)
    img[146:236, 20:110, :] = [40, 40, 220]
    # Yellow block (bottom-right, R+G)
    img[146:236, 146:236, :] = [230, 220, 30]
    # White circle (center)
    yy, xx = np.ogrid[:256, :256]
    mask = (yy - 128)**2 + (xx - 128)**2 < 30**2
    img[mask] = [255, 255, 255]
    return img

rgb_image = load_sample_image()

# Center crop to square and resize to 256x256 for consistency
h, w, c = rgb_image.shape
side = min(h, w)
y_start = (h - side) // 2
x_start = (w - side) // 2
rgb_image = rgb_image[y_start:y_start+side, x_start:x_start+side, :]
rgb_image = np.array(Image.fromarray(rgb_image).resize((256, 256)))

print(f"Tensor shape : {rgb_image.shape}  (H, W, C)")
print(f"Data type    : {rgb_image.dtype}")
print(f"Pixel range  : [{rgb_image.min()}, {rgb_image.max()}]")


### The Raw Loaded Image
Let's first look at the raw image we'll use throughout this tutorial:


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(rgb_image)
ax.set_title(f"Raw Loaded Image\nShape: {rgb_image.shape}, dtype: {rgb_image.dtype}", fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()


### 1.1 Synthesizing 1-Channel, 2-Channel, and 3-Channel Images


In [ ]:
# 1. One-Channel (Grayscale) — perceptual luminance
gray_1ch = (0.2989 * rgb_image[:,:,0].astype(np.float64)
          + 0.5870 * rgb_image[:,:,1].astype(np.float64)
          + 0.1140 * rgb_image[:,:,2].astype(np.float64))

# 2. Two-Channel (Gradient vector field)
grad_x = np.gradient(gray_1ch, axis=1)
grad_y = np.gradient(gray_1ch, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1-ch
im0 = axes[0].imshow(gray_1ch, cmap='gray')
axes[0].set_title(f"1-Channel (Grayscale)\nShape: {gray_1ch.shape}", fontsize=13)
axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# 2-ch — show gradient magnitude
grad_mag = np.sqrt(grad_x**2 + grad_y**2)
im1 = axes[1].imshow(grad_mag, cmap='inferno')
axes[1].set_title(f"2-Channel (Gradient Magnitude)\n" +
                  r"$\sqrt{(\partial I/\partial x)^2 + (\partial I/\partial y)^2}$" +
                  f"\nShape: ({grad_mag.shape[0]}, {grad_mag.shape[1]}, 2)", fontsize=13)
axes[1].axis('off')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# 3-ch
axes[2].imshow(rgb_image)
axes[2].set_title(f"3-Channel (Full RGB Color)\nShape: {rgb_image.shape}", fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()


## 1.2 Channel Decomposition & Pairwise Comparison

A common misunderstanding is that each channel of an RGB image is already a "colored" picture. In reality, **each individual channel is simply a 2D scalar intensity matrix**. 

To help bridge the intuition between raw numerical intensity and perceived visual color, each pairwise comparison displays:
- **Top row**: The channel's true chromatic appearance (e.g., Red shown as red-tinted color $[R, 0, 0]$, Green as $[0, G, 0]$, Blue as $[0, 0, B]$).
- **Middle row**: The underlying grayscale numerical intensity map (what the neural network actually computes over).
- **Bottom row**: The pixel value frequency histogram for that channel.


In [ ]:
R_channel = rgb_image[:, :, 0]
G_channel = rgb_image[:, :, 1]
B_channel = rgb_image[:, :, 2]

zeros_ch = np.zeros_like(R_channel)
R_tinted = np.stack([R_channel, zeros_ch, zeros_ch], axis=-1)
G_tinted = np.stack([zeros_ch, G_channel, zeros_ch], axis=-1)
B_tinted = np.stack([zeros_ch, zeros_ch, B_channel], axis=-1)

def compare_two_channels_vivid(ch1_data, ch1_tinted, ch1_name, ch1_color,
                               ch2_data, ch2_tinted, ch2_name, ch2_color):
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(3, 2, height_ratios=[3, 3, 1.5])

    # Row 1: Chromatic view
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(ch1_tinted)
    ax1.set_title(f"{ch1_name} Chromatic View [R,G,B]", fontsize=12, fontweight='bold')
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(ch2_tinted)
    ax2.set_title(f"{ch2_name} Chromatic View [R,G,B]", fontsize=12, fontweight='bold')
    ax2.axis('off')

    # Row 2: Grayscale scalar intensity
    ax3 = fig.add_subplot(gs[1, 0])
    im3 = ax3.imshow(ch1_data, cmap='gray', vmin=0, vmax=255)
    ax3.set_title(f"{ch1_name} Scalar Intensity (What Conv Sees)", fontsize=11)
    ax3.axis('off')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

    ax4 = fig.add_subplot(gs[1, 1])
    im4 = ax4.imshow(ch2_data, cmap='gray', vmin=0, vmax=255)
    ax4.set_title(f"{ch2_name} Scalar Intensity (What Conv Sees)", fontsize=11)
    ax4.axis('off')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

    # Row 3: Histograms
    ax5 = fig.add_subplot(gs[2, 0])
    ax5.hist(ch1_data.ravel(), bins=50, color=ch1_color, alpha=0.7)
    ax5.set_xlim([0, 255]); ax5.set_xlabel("Pixel Value (0-255)"); ax5.grid(alpha=0.3)

    ax6 = fig.add_subplot(gs[2, 1])
    ax6.hist(ch2_data.ravel(), bins=50, color=ch2_color, alpha=0.7)
    ax6.set_xlim([0, 255]); ax6.set_xlabel("Pixel Value (0-255)"); ax6.grid(alpha=0.3)

    fig.suptitle(f"Pairwise Channel Comparison: {ch1_name} vs {ch2_name}",
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

compare_two_channels_vivid(R_channel, R_tinted, "Red (R)", "red", G_channel, G_tinted, "Green (G)", "green")
compare_two_channels_vivid(R_channel, R_tinted, "Red (R)", "red", B_channel, B_tinted, "Blue (B)", "blue")
compare_two_channels_vivid(G_channel, G_tinted, "Green (G)", "green", B_channel, B_tinted, "Blue (B)", "blue")


### 1.3 Full Channel Synthesis: All Channels Reconstructed

Each channel can be isolated as a monochromatic component by zeroing out the other two channels:
$$\mathbf{I}_{\text{RGB}} = [R, 0, 0] + [0, G, 0] + [0, 0, B]$$


In [ ]:
# Work in float [0, 1] so matplotlib renders colours correctly
img_f = rgb_image.astype(np.float32) / 255.0
R_f, G_f, B_f = img_f[:,:,0], img_f[:,:,1], img_f[:,:,2]
z = np.zeros_like(R_f)

only_red   = np.stack([R_f, z, z], axis=-1)
only_green = np.stack([z, G_f, z], axis=-1)
only_blue  = np.stack([z, z, B_f], axis=-1)

# Verify additive reconstruction
reconstructed = np.clip(only_red + only_green + only_blue, 0, 1)
assert np.allclose(reconstructed, img_f, atol=1e-6), "Decomposition check failed!"

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, img, title in zip(axes,
        [only_red, only_green, only_blue, reconstructed],
        ["Isolated Red $[R, 0, 0]$",
         "Isolated Green $[0, G, 0]$",
         "Isolated Blue $[0, 0, B]$",
         "Reconstructed RGB"]):
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 1.4 Preprocessing and Transformative Effects

Before an image enters a neural network, it is almost never passed in raw $[0, 255]$ integer form.

### Standard Preprocessing Techniques:
* **Min-Max Rescaling**: $x_{\text{scaled}} = x / 255 \in [0, 1]$
* **Standardization**: $x_{\text{std}} = (x - \mu) / \sigma$
* **Perceptual Grayscale**: $Y = 0.2989R + 0.5870G + 0.1140B$


In [ ]:
img_scaled = rgb_image.astype(np.float32) / 255.0
img_std = (img_scaled - img_scaled.mean()) / (img_scaled.std() + 1e-7)

unweighted_gray = np.mean(img_scaled, axis=-1)
perceptual_gray = (0.2989 * img_scaled[:,:,0]
                 + 0.5870 * img_scaled[:,:,1]
                 + 0.1140 * img_scaled[:,:,2])
diff_gray = np.abs(perceptual_gray - unweighted_gray)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img_scaled)
axes[0].set_title(f"Normalised [0, 1]\nMean={img_scaled.mean():.2f}", fontsize=11)
axes[0].axis('off')

axes[1].imshow(perceptual_gray, cmap='gray')
axes[1].set_title("Perceptual Luminance Y\n0.299R + 0.587G + 0.114B", fontsize=11)
axes[1].axis('off')

axes[2].imshow(unweighted_gray, cmap='gray')
axes[2].set_title("Naive Mean (R+G+B)/3", fontsize=11)
axes[2].axis('off')

im3 = axes[3].imshow(diff_gray, cmap='magma')
axes[3].set_title(r"|Y$_{perceptual}$ - Y$_{naive}$|", fontsize=11)
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Separate histograms for each preprocessing stage
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
axes[0].hist(rgb_image.ravel(), bins=60, color='steelblue', alpha=0.8)
axes[0].set_title(f"Raw [0, 255]\nMean={rgb_image.mean():.1f}", fontweight='bold')
axes[0].set_xlabel("Value")
axes[0].grid(alpha=0.3)

axes[1].hist(img_scaled.ravel(), bins=60, color='darkorange', alpha=0.8)
axes[1].set_title(f"Rescaled [0, 1]\nMean={img_scaled.mean():.3f}", fontweight='bold')
axes[1].set_xlabel("Value")
axes[1].grid(alpha=0.3)

axes[2].hist(img_std.ravel(), bins=60, color='seagreen', alpha=0.8)
axes[2].set_title(f"Standardised\nMean={img_std.mean():.2f}, Std={img_std.std():.2f}", fontweight='bold')
axes[2].set_xlabel("Value")
axes[2].grid(alpha=0.3)

plt.suptitle("Pixel Value Distributions Under Preprocessing", fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


---
# Part 2: Convolutions — How Filters/Kernels Work

### Mathematical Formulation
$$S(i, j) = \sum_{m}\sum_{n} I(i+m,\; j+n)\; K(m, n) + b$$


In [ ]:
KERNELS = {
    "Identity": np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float32),
    "Box Blur": np.ones((3,3), dtype=np.float32) / 9.0,
    "Sobel Horiz": np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32),
    "Sobel Vert": np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
    "Laplacian": np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float32),
    "Sharpen": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
}

print(f"=== Successfully Loaded {len(KERNELS)} Classical Convolution Kernels ===")
for idx, (k_name, k_mat) in enumerate(KERNELS.items(), 1):
    print(f" [{idx}] {k_name:12s} ({k_mat.shape[0]}x{k_mat.shape[1]}) -> Center={k_mat[1,1]:5.2f}, Sum={k_mat.sum():5.2f}")


### 2.1 Pure NumPy 2D Convolution — No Black Boxes

> **Interactive Exploration**:  
> In the code below, every classical kernel in `KERNELS` is convolved with the grayscale image.  
> **Try this**: Define your own custom kernel in `KERNELS` (e.g. an Emboss filter `np.array([[-2,-1,0],[-1,1,1],[0,1,2]])` or a $5 \times 5$ Gaussian blur) and re-run to see how your kernel sculpts new visual features!


In [ ]:
def manual_conv2d(image_2d, kernel_2d, stride=1, padding='valid'):
    H, W = image_2d.shape
    Kh, Kw = kernel_2d.shape
    if padding == 'same':
        ph, pw = (Kh - 1) // 2, (Kw - 1) // 2
        image_2d = np.pad(image_2d, ((ph, ph), (pw, pw)), constant_values=0)
    pH, pW = image_2d.shape
    oH = (pH - Kh) // stride + 1
    oW = (pW - Kw) // stride + 1
    out = np.zeros((oH, oW), dtype=np.float32)
    for i in range(oH):
        for j in range(oW):
            r, c = i * stride, j * stride
            out[i, j] = np.sum(image_2d[r:r+Kh, c:c+Kw] * kernel_2d)
    return out

# Apply every classical kernel to the grayscale image
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for idx, (name, kern) in enumerate(KERNELS.items()):
    ax = axes.ravel()[idx]
    fmap = manual_conv2d(perceptual_gray.astype(np.float32), kern, padding='same')
    ax.imshow(fmap, cmap='gray')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()


### 2.2 Numerical Equivalence: NumPy vs TensorFlow `Conv2D`

Let's rigorously prove that TensorFlow's optimized C++/CUDA convolution engine produces the **exact same arithmetic results** as our manual pure-Python nested loop.


In [ ]:
test_kernel = KERNELS["Sobel Horiz"]
test_input = perceptual_gray.astype(np.float32)

manual_result = manual_conv2d(test_input, test_kernel, padding='same')

tf_img = tf.constant(test_input[None, :, :, None], dtype=tf.float32)
tf_flt = tf.constant(test_kernel[:, :, None, None], dtype=tf.float32)
tf_result = tf.squeeze(tf.nn.conv2d(tf_img, tf_flt, strides=1, padding='SAME')).numpy()

diff_matrix = np.abs(manual_result - tf_result)
err = np.max(diff_matrix)
mean_err = np.mean(diff_matrix)

print(f"=== Convolution Engine Equivalence Report ===")
print(f" 1. Spatial Dimension Match : Manual {manual_result.shape} vs TF {tf_result.shape} -> MATCH")
print(f" 2. Output Value Range      : [{manual_result.min():.4f}, {manual_result.max():.4f}]")
print(f" 3. Maximum Absolute Error  : {err:.2e} (tolerance: 1e-4)")
print(f" 4. Mean Absolute Error     : {mean_err:.2e}")
print(f" 5. Sample 3x3 Center Patch Comparison:")
print(f"    Manual center values:\n{np.round(manual_result[126:129, 126:129], 3)}")
print(f"    TF center values:\n{np.round(tf_result[126:129, 126:129], 3)}")

assert err < 1e-4, "Mismatch between NumPy and TensorFlow!"
print("\nVERDICT: NumPy manual conv matches TensorFlow Conv2D within floating-point precision.")


### 2.3 Multi-Channel Convolutions ($C_{in} > 1$)

When an input tensor has $C_{in}$ channels (e.g., $C=3$ for RGB), each convolution filter is a **3-dimensional tensor**:
$$\mathbf{K} \in \mathbb{R}^{K_h \times K_w \times C_{in}}$$

Each channel slice of the filter $K_{:,:,c}$ convolves with the corresponding image channel $I_{:,:,c}$. The $C_{in}$ resulting 2D maps are **summed element-wise**:
$$S(i, j) = \sum_{c=1}^{C_{in}} \left( I_c * K_c \right)(i, j)$$

> **Why is the Green channel contribution all zeros below?**  
> In this pedagogical demonstration, we deliberately constructed the filter with weights `filter_3d[:,:,1] = 0.0` for the green channel. This simulates a **channel-selective feature detector** (e.g., a filter sensitive to red and blue contrasts, ignoring green). Notice how the summed output reflects only the active channels!
>
> **Try Exploring**: Experiment with different per-channel filter combinations! For instance, try assigning a horizontal Sobel to Red, a vertical Sobel to Green, and a box blur to Blue to observe how complementary channels blend into a composite feature map.


In [ ]:
# 3D filter: strong Red Sobel, zero Green, negative-half Blue Sobel
# Feel free to tweak these channel assignments to observe different feature mixtures!
filter_3d = np.zeros((3, 3, 3), dtype=np.float32)
filter_3d[:,:,0] = KERNELS["Sobel Vert"]       # Red channel weights
filter_3d[:,:,1] = 0.0                          # Green channel weights (deliberately zero; try KERNELS['Box Blur']!)
filter_3d[:,:,2] = -0.5 * KERNELS["Sobel Vert"] # Blue channel weights

img_f32 = rgb_image.astype(np.float32) / 255.0

# Per-channel contributions
contrib_R = manual_conv2d(img_f32[:,:,0], filter_3d[:,:,0], padding='same')
contrib_G = manual_conv2d(img_f32[:,:,1], filter_3d[:,:,1], padding='same')
contrib_B = manual_conv2d(img_f32[:,:,2], filter_3d[:,:,2], padding='same')
summed    = contrib_R + contrib_G + contrib_B

# Use a shared colour range so comparisons are meaningful
vabs = max(np.abs(contrib_R).max(), np.abs(contrib_B).max(), np.abs(summed).max(), 0.01)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ["Red Contribution\n(Sobel Vert * R)",
          "Green Contribution\n(Zero Weights * G = 0)",
          "Blue Contribution\n(-0.5 * Sobel Vert * B)",
          "Summed Feature Map\n(R + G + B contributions)"]
data   = [contrib_R, contrib_G, contrib_B, summed]
for ax, d, t in zip(axes, data, titles):
    im = ax.imshow(d, cmap='coolwarm', vmin=-vabs, vmax=vabs)
    ax.set_title(t, fontsize=11, fontweight='bold')
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f"Red ch response range   : [{contrib_R.min():.3f}, {contrib_R.max():.3f}]")
print(f"Green ch response range : [{contrib_G.min():.3f}, {contrib_G.max():.3f}] (Identically zero)")
print(f"Blue ch response range  : [{contrib_B.min():.3f}, {contrib_B.max():.3f}]")
print(f"Summed response range   : [{summed.min():.3f}, {summed.max():.3f}]")


---
# Part 3: Spatial Mechanics — Patches, Padding, and Pooling

## 3.1 How Patches and Receptive Fields are Formed

A convolution layer does not inspect the whole image at once. Instead, it extracts **local spatial patches** (receptive fields) across the image. Below, we place $32 \times 32$ receptive field windows over key regions of our Jedi image:

> **Try Exploring**: Adjust the window size `ks` (e.g. try `ks = 16` for fine local details or `ks = 48` for wider context) or modify `patch_coords` to zoom into other interesting areas of the image!


In [ ]:
# Feel free to change ks (e.g., 16, 48) or adjust coordinate tuples (y, x, label)
ks = 32
patch_coords = [
    (75, 105, "P1: Face / Ears"),
    (115, 155, "P2: Lightsaber"),
    (155, 95, "P3: Jedi Robe"),
    (25, 25, "P4: Background"),
]
colors_list = ['red', 'lime', 'cyan', 'magenta']

fig = plt.figure(figsize=(16, 5))
ax_main = fig.add_subplot(1, 6, (1, 2))
ax_main.imshow(rgb_image)
for idx, (y, x, label) in enumerate(patch_coords):
    rect = plt.Rectangle((x, y), ks, ks, lw=2.5, edgecolor=colors_list[idx], facecolor='none')
    ax_main.add_patch(rect)
    ax_main.text(x, y - 4, f"P{idx+1}", color=colors_list[idx], fontweight='bold', fontsize=11)
ax_main.set_title("Full Image with Receptive Field Windows", fontweight='bold')
ax_main.axis('off')

for idx, (y, x, label) in enumerate(patch_coords):
    ax = fig.add_subplot(1, 6, 3 + idx)
    patch = rgb_image[y:y+ks, x:x+ks, :]
    ax.imshow(patch)
    ax.set_title(f"{label}\n({ks}x{ks})", color=colors_list[idx], fontweight='bold', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 3.2 Padding: Valid vs. Same

The formula for the spatial output dimension is:
$$W_{\text{out}} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

To visually appreciate the difference, below we plot all three on a **shared coordinate grid**:
- **Valid (no padding)**: Spatial resolution shrinks by $(K - 1)$ pixels.
- **Same (zero padding)**: Padded with zeros so output spatial resolution exactly matches input resolution.

> **Try Exploring**: Change the kernel dimension `k = 5` to `k = 7` or `k = 9` in the code below to see how larger kernels amplify boundary shrinkage under `'valid'` padding!


In [ ]:
# Feel free to change k (e.g. k = 3, 7, 9) to observe different shrinkage amounts!
inp_h, inp_w = 28, 28
k = 5
dummy = np.random.rand(inp_h, inp_w).astype(np.float32)
ones_k = np.ones((k, k), dtype=np.float32)

valid_out = manual_conv2d(dummy, ones_k, padding='valid')
same_out  = manual_conv2d(dummy, ones_k, padding='same')

# Embedded valid map in a 28x28 canvas to visualize physical shrinkage
valid_embedded = np.full((inp_h, inp_w), np.nan, dtype=np.float32)
vh, vw = valid_out.shape
valid_embedded[:vh, :vw] = valid_out

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(dummy, cmap='viridis')
axes[0].set_title(f"1. Original Input\nShape: ({inp_h}, {inp_w})", fontweight='bold')
axes[0].set_xticks([0, 7, 14, 21, 27]); axes[0].set_yticks([0, 7, 14, 21, 27])

im1 = axes[1].imshow(valid_embedded, cmap='viridis')
axes[1].set_title(f"2. 'Valid' Conv Output (No Padding)\nShape: ({vh}, {vw}) — Shrunk by {inp_h-vh}px!", fontweight='bold', color='darkred')
axes[1].set_xticks([0, 7, 14, 21, 27]); axes[1].set_yticks([0, 7, 14, 21, 27])

axes[2].imshow(same_out, cmap='viridis')
axes[2].set_title(f"3. 'Same' Conv Output (Zero-Padded)\nShape: {same_out.shape} — Preserved!", fontweight='bold', color='darkgreen')
axes[2].set_xticks([0, 7, 14, 21, 27]); axes[2].set_yticks([0, 7, 14, 21, 27])

plt.suptitle("Visualising Physical Boundary Shrinkage in Convolutions", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Input spatial resolution   : {inp_h} x {inp_w}")
print(f"'Valid' output resolution  : {valid_out.shape[0]} x {valid_out.shape[1]} (lost {inp_h - valid_out.shape[0]} rows/cols)")
print(f"'Same' output resolution   : {same_out.shape[0]} x {same_out.shape[1]} (dimensions fully preserved)")


## 3.3 Pooling: Types and Visual Effects

| Type | Formula | Effect |
|:---|:---|:---|
| **Max Pool** | $y = \max(\text{window})$ | Selects the peak activation; preserves prominent edges, suppress noise |
| **Avg Pool** | $y = \text{mean}(\text{window})$ | Averages all values; diffuses/blurs boundaries into a smooth summary |
| **Global Avg Pool** | $(H,W,C) \to (1,C)$ | Collapses spatial dims into a single feature vector |

> **Understanding the Visual Output**:  
> In natural edge maps, a large portion of pixels are near-zero (background) while true edges are bright and sparse.  
> - **Max Pooling** retains the bright edge peaks with sharp contrast.  
> - **Average Pooling** dilutes edge intensities with surrounding background zeros, creating a lower-intensity, blurred/diffused appearance across regions.


In [ ]:
edge_raw = manual_conv2d(perceptual_gray.astype(np.float32),
                         KERNELS["Sobel Vert"], padding='same')
edge_abs = np.abs(edge_raw)

edge_tf = tf.constant(edge_abs[None, :, :, None], dtype=tf.float32)
max_pooled = tf.squeeze(tf.nn.max_pool2d(edge_tf, ksize=2, strides=2, padding='VALID')).numpy()
avg_pooled = tf.squeeze(tf.nn.avg_pool2d(edge_tf, ksize=2, strides=2, padding='VALID')).numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(edge_abs, cmap='gray')
axes[0].set_title(f"Original Edge Map\nShape: {edge_abs.shape}\nMean: {edge_abs.mean():.3f}, Max: {edge_abs.max():.2f}", fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(max_pooled, cmap='gray')
axes[1].set_title(f"Max Pooling (2x2, stride 2)\nShape: {max_pooled.shape} — Sharp Edges Kept\nMean: {max_pooled.mean():.3f}, Max: {max_pooled.max():.2f}", fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(avg_pooled, cmap='gray')
axes[2].set_title(f"Avg Pooling (2x2, stride 2)\nShape: {avg_pooled.shape} — Diffused / Smoothed\nMean: {avg_pooled.mean():.3f}, Max: {avg_pooled.max():.2f}", fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Max-pool pixel range : [{max_pooled.min():.2f}, {max_pooled.max():.2f}]")
print(f"Avg-pool pixel range : [{avg_pooled.min():.2f}, {avg_pooled.max():.2f}]")
print(f"Max-pool mean ({max_pooled.mean():.3f}) > Avg-pool mean ({avg_pooled.mean():.3f}) -> Confirms peak preservation.")


---
# Part 4: The Hierarchy of Features — Why Depth & Non-Linearity Matter

### The Collapse Theorem
Without activation functions, any deep network collapses to a single linear map:
$$W_3(W_2(W_1 x)) = W_{\text{eff}}\, x$$

### ReLU breaks linearity:
$$\text{ReLU}(z) = \max(0, z)$$


In [ ]:
raw_conv = manual_conv2d(perceptual_gray.astype(np.float32),
                       KERNELS["Sobel Horiz"], padding='same')
relu_conv = np.maximum(0, raw_conv)

vabs_r = np.max(np.abs(raw_conv))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

im0 = axes[0].imshow(raw_conv, cmap='coolwarm', vmin=-vabs_r, vmax=vabs_r)
axes[0].set_title("Pre-ReLU (linear output)\nPositive & negative values", fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(relu_conv, cmap='hot')
axes[1].set_title("Post-ReLU: max(0, x)\nNegatives gated to 0", fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].hist(raw_conv.ravel(), bins=50, color='blue', alpha=0.5, label='Pre-ReLU')
axes[2].hist(relu_conv.ravel(), bins=50, color='red', alpha=0.5, label='Post-ReLU')
axes[2].set_title("Activation Distribution", fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

sparsity = np.mean(relu_conv == 0) * 100
print(f"Sparsity after ReLU: {sparsity:.1f}% of activations are zero")


## 4.2 Visualizing Feature Maps Across Layers

* **Layer 1**: Edges, gradients, textures (high spatial resolution).
* **Layer 2**: Corners, intersections (medium resolution).
* **Layer 3+**: Abstract semantic patterns (low resolution).


In [ ]:
inputs = tf.keras.Input(shape=(256, 256, 3))
x1 = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu', name='conv1')(inputs)
p1 = tf.keras.layers.MaxPool2D(2, name='pool1')(x1)
x2 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv2')(p1)
p2 = tf.keras.layers.MaxPool2D(2, name='pool2')(x2)
x3 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu', name='conv3')(p2)

extractor = tf.keras.Model(inputs, [x1, p1, x2, p2, x3])
batch = img_f32[None, :, :, :]
a_c1, a_p1, a_c2, a_p2, a_c3 = extractor.predict(batch, verbose=0)

print("=== Sequential Deep CNN Architecture & Activation Flow ===")
print(f" [Input]  Tensor Shape          : {batch.shape}")
print(f" |")
print(f" +--> [Stage 1 - Conv2D]        : {a_c1.shape}  (16 channels, preserves 256x256 resolution)")
print(f" |    +--> [Stage 1 - MaxPool2D]: {a_p1.shape}  (downsampled 2x -> 128x128)")
print(f" |")
print(f" +--> [Stage 2 - Conv2D]        : {a_c2.shape}  (32 channels, 128x128 resolution)")
print(f" |    +--> [Stage 2 - MaxPool2D]: {a_p2.shape}  (downsampled 2x -> 64x64)")
print(f" |")
print(f" +--> [Stage 3 - Conv2D]        : {a_c3.shape}  (64 deep abstraction channels, 64x64)")


In [ ]:
def plot_layer(activations, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(18, 3))
    for i in range(n):
        axes[i].imshow(activations[0, :, :, i], cmap='viridis')
        axes[i].set_title(f"Ch {i+1}")
        axes[i].axis('off')
    fig.suptitle(f"{title}  (spatial {activations.shape[1]}x{activations.shape[2]})",
                 fontsize=13, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

print(">>> STAGE 1: Low-level primitives (Conv 1)")
plot_layer(a_c1, "Stage 1 — Early Conv Layer")
print(">>> STAGE 2: Intermediate motifs (Conv 2)")
plot_layer(a_c2, "Stage 2 — Middle Conv Layer")
print(">>> STAGE 3: Deep abstractions (Conv 3)")
plot_layer(a_c3, "Stage 3 — Deep Conv Layer")


---
# Part 5: Interactive Exercises (TODOs)

Complete the exercises below. Replace the `None` or `# ???` placeholders with your answers.

> **Pedagogical Note for Learners**:  
> In exercises that produce visual outputs (such as **Exercise 1** and **Exercise 3**), there are **no rigid assertions**.  
> Once your baseline solution works, **feel free to experiment freely**! Try varying numerical coefficients, changing signs, and combining weights to observe how the visual output responds in real time.

---

### Exercise 1: Custom Diagonal Edge Detector

Design a $3 \times 3$ kernel that detects $45^\circ$ diagonal edges:
$$K_{45} = \begin{bmatrix} 0 & 1 & 2 \\ -1 & 0 & 1 \\ -2 & -1 & 0 \end{bmatrix}$$

**Task**: Fill in the three rows of the kernel matrix below.

> **Exploration**: Once you observe the standard $45^\circ$ response, try inverting or transposing the matrix (e.g. creating a $135^\circ$ edge detector) to observe how the filter's orientation sensitivity rotates!


In [ ]:
# TODO: Fill in each row of the 3x3 diagonal edge kernel.
# Hint: top-right corner has the largest positive value (2),
#        bottom-left corner has the largest negative value (-2).
#
# (Feel free to experiment with other angle combinations after testing!)
diagonal_kernel = np.array([
    # [  ?,  ?,  ?],   # row 0  — hint: starts at 0, ends at 2
    # [  ?,  ?,  ?],   # row 1  — hint: center is 0
    # [  ?,  ?,  ?],   # row 2  — hint: starts at -2, ends at 0
], dtype=np.float32)

# --- Uncomment the block below once you have filled in the kernel ---
# diag_response = manual_conv2d(perceptual_gray.astype(np.float32),
#                               diagonal_kernel, padding='same')
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# axes[0].imshow(perceptual_gray, cmap='gray')
# axes[0].set_title("Input Grayscale", fontweight='bold'); axes[0].axis('off')
# axes[1].imshow(np.abs(diag_response), cmap='inferno')
# axes[1].set_title("45-Degree Diagonal Edge Response", fontweight='bold'); axes[1].axis('off')
# plt.tight_layout(); plt.show()


---
### Exercise 2: Spatial Feature Map Dimension Calculator

When designing convolutional neural network architectures, you must determine the spatial dimensions $(H_{\text{out}}, W_{\text{out}})$ produced by each layer.

The formula for the 1D spatial output dimension given an input dimension $W$, kernel size $K$, padding $P$, and stride $S$ is:
$$W_{\text{out}} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

**Your Objective**:
Implement `compute_conv_output_dim(input_dim, kernel_size, padding_amount, stride)` to compute this formula dynamically.

**Expected Test Outcomes**:
1. ResNet Layer 1: $W=224, K=7, P=3, S=2 \implies \lfloor(224 - 7 + 6)/2\rfloor + 1 = 112$
2. MNIST Valid Conv: $W=28, K=3, P=0, S=1 \implies \lfloor(28 - 3 + 0)/1\rfloor + 1 = 26$
3. High-res Input: $W=768, K=5, P=2, S=2 \implies \lfloor(768 - 5 + 4)/2\rfloor + 1 = 384$

*(Note: This exercise includes automated assertion tests to verify your implementation)*


In [ ]:
def compute_conv_output_dim(input_dim, kernel_size, padding_amount, stride):
    """Compute output spatial dimension given input, kernel, padding, and stride."""
    # TODO: Translate the formula into Python:
    # output_dim = ((input_dim - kernel_size + 2 * padding_amount) // stride) + 1
    output_dim = None  # ???
    return output_dim

# --- Test cases (these will verify your formula dynamically) ---
try:
    t1 = compute_conv_output_dim(224, 7, 3, 2)
    assert t1 == 112, f"Test 1 failed: expected 112, got {t1}"

    t2 = compute_conv_output_dim(28, 3, 0, 1)
    assert t2 == 26, f"Test 2 failed: expected 26, got {t2}"

    t3 = compute_conv_output_dim(768, 5, 2, 2)
    assert t3 == 384, f"Test 3 failed: expected 384, got {t3}"

    print("SUCCESS: All dimension tests passed!")

    # Visual summary of what each configuration produces
    configs = [("ResNet L1", 224, 7, 3, 2, 112),
               ("MNIST Valid", 28, 3, 0, 1, 26),
               ("High-Res (768px)", 768, 5, 2, 2, 384)]
    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    for ax, (label, W, K, P, S, expected) in zip(axes, configs):
        ax.barh(["Input", "Output"], [W, expected], color=['steelblue', 'darkorange'])
        ax.set_title(f"{label}\nW={W}, K={K}, P={P}, S={S}", fontweight='bold')
        ax.set_xlabel("Spatial dimension")
    plt.tight_layout()
    plt.show()
except (AssertionError, TypeError) as e:
    print(f"Not yet correct: {e}")


---
### Exercise 3: Channel Attention — Isolate Red Objects

Create a weight vector $\mathbf{w} = [w_R, w_G, w_B]$ that amplifies red while suppressing green and blue, then apply ReLU.

**Hint**: Use a positive weight for Red and negative weights for Green and Blue (e.g., $w_R = 1.0, w_G = -0.5, w_B = -0.5$).

> **Sandbox Exploration (Open-Ended)**:  
> Because this exercise has **no strict assertion check**, you are encouraged to play around with different weight combinations!  
> - What happens if you prioritize Green: $[w_R = -0.5, w_G = 1.2, w_B = -0.5]$?  
> - What if you isolate Blue/Cyan (the lightsaber energy): $[w_R = -0.8, w_G = 0.5, w_B = 1.5]$?  
> Test different values to see how the attention mask shifts across the image!


In [ ]:
# TODO: Set three weights [w_R, w_G, w_B].
# Baseline hint: try w_R > 0, w_G < 0, w_B < 0 (e.g., 1.0, -0.5, -0.5)
# After observing the result, feel free to try other ratios to isolate green or blue features!
w_R = None  # ???
w_G = None  # ???
w_B = None  # ???

# --- Uncomment the block below once you have set the weights ---
# attention = (w_R * img_f32[:,:,0] + w_G * img_f32[:,:,1] + w_B * img_f32[:,:,2])
# attention = np.maximum(0, attention)  # ReLU gate
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# axes[0].imshow(rgb_image)
# axes[0].set_title("Original RGB", fontweight='bold'); axes[0].axis('off')
# axes[1].imshow(attention, cmap='hot')
# axes[1].set_title(f"Channel Attention Mask\nWeights: [{w_R}, {w_G}, {w_B}]", fontweight='bold'); axes[1].axis('off')
# plt.tight_layout(); plt.show()


---
### Exercise 4: Manual $2 \times 2$ Max Pooling

Implement the max-pooling selection for a single $2 \times 2$ window.

**Task**: Return the single maximum numerical value within a 2x2 NumPy matrix.

*(Note: This function is verified by an automated test case below)*


In [ ]:
def manual_max_pool_2x2(patch_2x2):
    """Return the maximum value in a 2x2 numpy array."""
    # TODO: return the maximum value using np.max(patch_2x2)
    return None  # ???

# --- Test ---
try:
    sample = np.array([[1.5, 4.2], [9.8, 3.1]], dtype=np.float32)
    result = manual_max_pool_2x2(sample)
    assert result == 9.8, f"Expected 9.8, got {result}"
    print(f"SUCCESS: max_pool_2x2({sample.tolist()}) = {result}")

    # Visual: apply your function across a small feature map
    small_fm = np.random.rand(4, 4).astype(np.float32) * 10
    pooled = np.zeros((2, 2), dtype=np.float32)
    for r in range(2):
        for c in range(2):
            pooled[r, c] = manual_max_pool_2x2(small_fm[r*2:r*2+2, c*2:c*2+2])
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    axes[0].imshow(small_fm, cmap='YlOrRd'); axes[0].set_title("4x4 Input")
    for i in range(4):
        for j in range(4):
            axes[0].text(j, i, f"{small_fm[i,j]:.1f}", ha='center', va='center', fontsize=9)
    axes[1].imshow(pooled, cmap='YlOrRd'); axes[1].set_title("2x2 Max-Pooled")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{pooled[i,j]:.1f}", ha='center', va='center', fontsize=11)
    plt.tight_layout(); plt.show()
except (AssertionError, TypeError) as e:
    print(f"Not yet correct: {e}")


---
## Summary and Key Takeaways

1. **Images = Tensors** of shape $(H, W, C)$. Channels represent independent measurements.
2. **Preprocessing** (normalisation, standardisation) stabilises gradient flow.
3. **Convolutions** slide a small kernel across local patches (element-wise multiply + sum).
4. **Multi-channel filters** aggregate across all $C_{in}$ channels into a single 2D map.
5. **Padding**: `same` preserves spatial size; `valid` shrinks.
6. **Pooling**: Max preserves peak activations; Average diffuses into a smoothed summary.
7. **Non-linearity (ReLU)** is essential — without it, deep networks collapse to linear regression.
8. **Early layers** detect edges; **deep layers** encode abstract, semantic features.
